# PDF to Chroma Pipeline

This notebook loads a PDF, splits it into chunks, stores the chunks in Chroma, and runs a retrieval query.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\sujat\AppData\Local\Temp\ipykernel_11008\2634496628.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\sujat\projects\AI-Main\Advanced_Rag_Codes\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Prepare Paths and Configuration

In [2]:
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

project_root

WindowsPath('c:/Users/sujat/projects/AI-Main/Advanced_Rag_Codes/04_vector_stores')

In [3]:
env_path = project_root / ".env"
pdf_path = project_root / "documents" / "beyond-chatbots-ai-agents-next-real-shift.pdf"
persist_directory = project_root / "notebooks" / "chroma_langchain_db"
collection_name = "rag-pipeline"

print(f"PDF path: {pdf_path}")
print(f"Collection name: {collection_name}")
print(f"Persist directory: {persist_directory}")

PDF path: c:\Users\sujat\projects\AI-Main\Advanced_Rag_Codes\04_vector_stores\documents\beyond-chatbots-ai-agents-next-real-shift.pdf
Collection name: rag-pipeline
Persist directory: c:\Users\sujat\projects\AI-Main\Advanced_Rag_Codes\04_vector_stores\notebooks\chroma_langchain_db


In [4]:
# load_dotenv(dotenv_path=env_path)

# if not os.getenv("OPENAI_API_KEY"):
#     raise ValueError("Please add your OPENAI_API_KEY to the .env file before running this notebook.")

# print(f"Loaded environment from: {env_path}")

In [5]:
# Reuse the same embedding model as the other notebooks in this repo.
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("Embedding model is ready.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7964.41it/s]


Embedding model is ready.


## 2. Add Small Display Helpers

In [20]:
def preview_text(text, limit=120):
    """Return a short preview for cleaner notebook output."""
    if len(text) <= limit:
        return text
    return text[:limit] + "..."


def print_documents(title, docs):
    """Print retrieved documents using page metadata and a text preview."""
    print(title)
    for index, doc in enumerate(docs, start=1):
        print(f"{index}. page={doc.metadata.get('page')} | source={doc.metadata.get('source')}")
        print(f"   content={doc.page_content}")
    print()

## 3. Load the PDF

In [21]:
loader = PyPDFLoader(str(pdf_path))

In [22]:
docs = loader.load()
print(f"Total pages loaded: {len(docs)}")

incorrect startxref pointer(1)
parsing for Object Streams


Total pages loaded: 6


In [23]:
docs[0].page_content

'Page 1\n Beyond Chatbots: Why AI Agents Feel Like the\n Next Real Shift\nA practical long-form blog on planning, memory, tools, and retrieval in modern AI systems\nBy Editorial Desk\nThe moment AI stopped feeling like a demo\nFor a long time, the most common experience with AI felt theatrical. You typed a question, the model\nanswered in polished language, and for a moment it seemed almost magical. Then the illusion broke. Ask a\nfollow-up that required memory, factual grounding, or a small sequence of actions, and the system often fell\napart. It could sound confident without being connected to anything real. That gap between fluency and\nusefulness is exactly where AI agents enter the picture.\nAn AI agent is interesting not because it sounds human, but because it behaves like software with intent. It\ncan take a goal, figure out what it needs in order to make progress, and work through a sequence of steps\ninstead of improvising a single reply. In the simplest form, that might mean

In [10]:
print(f"First page preview: {preview_text(docs[0].page_content)}")
print(f"First page metadata: {docs[0].metadata}")

First page preview: Page 1
 Beyond Chatbots: Why AI Agents Feel Like the
 Next Real Shift
A practical long-form blog on planning, memory, to...
First page metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-03-12T20:36:07+05:00', 'author': 'By Editorial Desk', 'keywords': '', 'moddate': '2026-03-12T20:36:07+05:00', 'subject': '(unspecified)', 'title': 'Beyond Chatbots: Why AI Agents Feel Like the Next Real Shift', 'trapped': '/False', 'source': 'c:\\Users\\sujat\\projects\\AI-Main\\Advanced_Rag_Codes\\04_vector_stores\\documents\\beyond-chatbots-ai-agents-next-real-shift.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}


## 4. Split the PDF into Chunks

In [11]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
)

In [12]:
chunked_docs = text_splitter.split_documents(docs)
print(f"Total chunks created: {len(chunked_docs)}")

Total chunks created: 88


In [13]:
print(f"First chunk preview: {preview_text(chunked_docs[0].page_content)}")
print(f"First chunk metadata: {chunked_docs[0].metadata}")

First chunk preview: Page 1
 Beyond Chatbots: Why AI Agents Feel Like the
 Next Real Shift
A practical long-form blog on planning, memory, to...
First chunk metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-03-12T20:36:07+05:00', 'author': 'By Editorial Desk', 'keywords': '', 'moddate': '2026-03-12T20:36:07+05:00', 'subject': '(unspecified)', 'title': 'Beyond Chatbots: Why AI Agents Feel Like the Next Real Shift', 'trapped': '/False', 'source': 'c:\\Users\\sujat\\projects\\AI-Main\\Advanced_Rag_Codes\\04_vector_stores\\documents\\beyond-chatbots-ai-agents-next-real-shift.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}


## 5. Store Chunks in Chroma

In [14]:
collection_name

'rag-pipeline'

In [15]:
persist_directory

WindowsPath('c:/Users/sujat/projects/AI-Main/Advanced_Rag_Codes/04_vector_stores/notebooks/chroma_langchain_db')

In [16]:
vector_store = Chroma.from_documents(
    documents=chunked_docs,
    embedding=embeddings,
    collection_name=collection_name,
    persist_directory=str(persist_directory),
)

print(f"Stored {len(chunked_docs)} chunks in the '{collection_name}' collection.")

Stored 88 chunks in the 'rag-pipeline' collection.


## 6. Retrieve Relevant Chunks

In [17]:
query = "How do AI agents use tools and memory?"
query

'How do AI agents use tools and memory?'

In [18]:
results = vector_store.similarity_search(query, k=3)

print(f"Query: {query}\n")
print_documents("Retrieved chunks:", results)

Query: How do AI agents use tools and memory?

Retrieved chunks:
1. page=2 | source=c:\Users\sujat\projects\AI-Main\Advanced_Rag_Codes\04_vector_stores\documents\beyond-chatbots-ai-agents-next-real-shift.pdf
   content=reveals the reasoning surface of the system. The answer no longer feels like a black box. It feels like the
product of a process that can be inspected.
Why tools make agents actually useful
If memory gives an agent context, tools give it reach. A model can describe a search, but a tool can perform
2. page=0 | source=c:\Users\sujat\projects\AI-Main\Advanced_Rag_Codes\04_vector_stores\documents\beyond-chatbots-ai-agents-next-real-shift.pdf
   content=participate in real workflows.
This is also why vector stores and retrieval have become such central topics in modern AI tutorials. Once you
accept that an agent should gather context instead of guessing from memory alone, you need a mechanism to
3. page=5 | source=c:\Users\sujat\projects\AI-Main\Advanced_Rag_Codes\04_vector_s

In [24]:
retrieved_docs = vector_store.similarity_search_with_score(query, k=4)

for doc, score in retrieved_docs:
    print(f"Score: {score:.4f}")
    print(f"Content preview: {doc.page_content}")
    print(f"page_no. {doc.metadata.get("page_label")}")
    print()

Score: 0.4684
Content preview: reveals the reasoning surface of the system. The answer no longer feels like a black box. It feels like the
product of a process that can be inspected.
Why tools make agents actually useful
If memory gives an agent context, tools give it reach. A model can describe a search, but a tool can perform
page_no. 3

Score: 0.6255
Content preview: participate in real workflows.
This is also why vector stores and retrieval have become such central topics in modern AI tutorials. Once you
accept that an agent should gather context instead of guessing from memory alone, you need a mechanism to
page_no. 1

Score: 0.6822
Content preview: into real work. AI agents are moving in that direction. And the more we build them around retrieval, structure,
and accountability, the more likely they are to stay there.
page_no. 6

Score: 0.7423
Content preview: An AI agent is interesting not because it sounds human, but because it behaves like software with intent. It
can take a go